# Tutorial 3-1: Similarity and Distance Measures
**Course:** CSEN 140: Machine Learning/Data Mining  
**Instructor:** Dr. David C. Anastasiu

## Objective
Proximity is the cornerstone of many machine learning algorithms, particularly clustering and instance-based learning. In this tutorial, we will implement and compare the mathematical measures discussed in lecture:
1. **Minkowski Distances:** Exploring the L1 (Manhattan), L2 (Euclidean), and L-infinity norms.
2. **The Scaling Problem:** Demonstrating how magnitude-heavy attributes can 'drown out' important features.
3. **Mahalanobis Distance:** Handling correlated data where Euclidean distance fails.
4. **Angular Similarity:** Using Cosine similarity for high-dimensional data like text.
5. **Binary Proximity:** Comparing Simple Matching (SMC) and Jaccard coefficients.

## 1. Minkowski Distance Family
Minkowski distance is a generalization of distance in a vector space. By changing the parameter $r$, we can calculate different norms:
- $r=1$: Manhattan (City Block) Distance.
- $r=2$: Euclidean Distance.
- $r \to \infty$: Supremum Distance (Maximum difference in any dimension).

In [ ]:
import numpy as np
import pandas as pd
from scipy.spatial import distance

# Define two points based on the example in Slide 13
p1 = np.array([0, 2])
p2 = np.array([2, 0])

print(f"Manhattan (L1): {distance.cityblock(p1, p2)}")
print(f"Euclidean (L2): {distance.euclidean(p1, p2):.3f}")
print(f"Supremum (L-inf): {distance.chebyshev(p1, p2)}")

## 2. The Importance of Scaling
As we discussed in class, large-magnitude attributes like Income ($10,000–$1,000,000) will dominate small-magnitude ones like Height (1.5m–1.8m) in distance calculations. 

Let's see how a small change in a large attribute affects Euclidean distance compared to a large change in a small attribute.

In [ ]:
# Person: [Height(m), Income($)]
person_a = np.array([1.7, 50000])
person_b = np.array([1.7, 50100]) # $100 difference
person_c = np.array([1.1, 50000]) # Massive 0.6m height difference

dist_ab = distance.euclidean(person_a, person_b)
dist_ac = distance.chebyshev(person_a, person_c)

print(f"Distance A to B (Income change): {dist_ab}")
print(f"Distance A to C (Height change): {dist_ac}")
print("The income difference completely masks the height difference!")

## 3. Mahalanobis Distance
Euclidean distance assumes that attributes are independent and have the same variance. If attributes are correlated (like the scatter plot on Slide 19), we use Mahalanobis distance, which accounts for the **covariance matrix**.

In [ ]:
# Generate correlated data
mean = [0, 0]
cov = [[1, 0.8], [0.8, 1]] # High correlation
data = np.random.multivariate_normal(mean, cov, 1000)

inv_cov = np.linalg.inv(np.cov(data.T))
u = np.array([1, 1])
v = np.array([-1, 1])

print(f"Euclidean u-v: {distance.euclidean(u, v):.3f}")
print(f"Mahalanobis u-v: {distance.mahalanobis(u, v, inv_cov):.3f}")
print("Notice how Mahalanobis distance is larger for points that go against the correlation trend.")

## 4. Binary Similarity: SMC vs. Jaccard
For binary data, **Simple Matching Coefficient (SMC)** counts both 1-1 and 0-0 matches as similarity. **Jaccard Similarity** only looks at 1-1 matches, making it ideal for sparse data (like products in a shopping cart) where 0-0 matches are not meaningful.

In [ ]:
from sklearn.metrics import jaccard_score

# Binary vectors from Slide 24
p = [1, 0, 0, 0, 0, 0, 0, 0, 0, 0]
q = [0, 0, 0, 0, 0, 0, 1, 0, 0, 1]

def smc(a, b):
    a, b = np.array(a), np.array(b)
    return np.sum(a == b) / len(a)

print(f"SMC: {smc(p, q)}")
print(f"Jaccard: {jaccard_score(p, q)}")